# 01. Cohort Base (기본 코호트)

## 목적
슬라이딩 윈도우 적용 **전** 기본 코호트 정의

## 포함 기준
- 성인 (18세 이상)
- ~~첫 번째 ICU 입실~~ → **모든 ICU 입실**
- 24시간 이상 체류

## 주의사항
- 같은 환자가 여러 stay로 포함될 수 있음
- Train/Val/Test split 시 **patient 단위**로 분리 필수 (Data Leakage 방지)

## 출력
- `cohort_base.csv`: ICU stay 정보 (1 row = 1 stay)

## 파이프라인 위치
```
01_cohort_base.ipynb  ← 현재
    ↓
02_vital_raw.ipynb, 03_lab_raw.ipynb, ...
    ↓
10_sliding_window_merge.ipynb (슬라이딩 윈도우 + 통합)
    ↓
11_preprocessing.ipynb
    ↓
12_feature_engineering.ipynb
```

In [ ]:
import duckdb
import pandas as pd
import os
from datetime import datetime

# 설정
DB_PATH = '../data/duckdb/mimic_total.duckdb'
OUTPUT_DIR = '../data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DuckDB 연결
con = duckdb.connect(DB_PATH)
print("=== 01. Cohort Base 생성 시작 ===")

## Step 1: 기본 코호트 정의

In [ ]:
print("Step 1: 기본 포함 기준 적용")
print("  - 성인 (18세 이상)")
print("  - 모든 ICU 입실 (재입실 포함)")
print("  - 24시간 이상 체류\n")

cohort_query = """
WITH all_stays AS (
    SELECT 
        i.subject_id,
        i.hadm_id,
        i.stay_id,
        CAST(i.intime AS TIMESTAMP) as intime,
        CAST(i.outtime AS TIMESTAMP) as outtime,
        CAST(i.los AS DOUBLE) as los,
        i.first_careunit,
        i.last_careunit,
        CAST(p.anchor_age AS INTEGER) as anchor_age,
        p.gender,
        CAST(p.dod AS TIMESTAMP) as dod,
        CAST(a.admittime AS TIMESTAMP) as admittime,
        CAST(a.dischtime AS TIMESTAMP) as dischtime,
        CAST(a.deathtime AS TIMESTAMP) as deathtime,
        a.hospital_expire_flag,
        -- 입실 순서 (재입실 여부 파악용)
        ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) as icu_seq
    FROM icustays i
    INNER JOIN patients p ON i.subject_id = p.subject_id
    INNER JOIN admissions a ON i.hadm_id = a.hadm_id
    WHERE 
        CAST(p.anchor_age AS INTEGER) >= 18
        AND CAST(i.los AS DOUBLE) >= 1.0
)
SELECT 
    subject_id,
    hadm_id,
    stay_id,
    intime,
    outtime,
    los,
    first_careunit,
    last_careunit,
    anchor_age,
    gender,
    dod,
    admittime,
    dischtime,
    deathtime,
    hospital_expire_flag,
    icu_seq,
    
    -- 재입실 여부 (피처로 활용 가능)
    CASE WHEN icu_seq > 1 THEN 1 ELSE 0 END as is_readmission,
    
    -- ICU 사망 여부
    CASE 
        WHEN deathtime IS NOT NULL AND deathtime <= outtime
        THEN 1 ELSE 0
    END as icu_mortality,
    
    -- 병원 사망 여부
    CASE 
        WHEN hospital_expire_flag = '1' THEN 1
        ELSE 0
    END as hospital_mortality
    
FROM all_stays
ORDER BY subject_id, icu_seq
"""

df_cohort = con.execute(cohort_query).df()

print(f"✓ 기본 코호트 생성 완료")
print(f"  - 총 ICU stay 수: {len(df_cohort):,}개")
print(f"  - 총 환자 수: {df_cohort['subject_id'].nunique():,}명")
print(f"  - 첫 입실: {(df_cohort['is_readmission'] == 0).sum():,}개 ({(df_cohort['is_readmission'] == 0).mean()*100:.1f}%)")
print(f"  - 재입실: {(df_cohort['is_readmission'] == 1).sum():,}개 ({(df_cohort['is_readmission'] == 1).mean()*100:.1f}%)")
print(f"  - ICU 사망: {df_cohort['icu_mortality'].sum():,}개 ({df_cohort['icu_mortality'].mean()*100:.2f}%)")
print(f"  - 병원 사망: {df_cohort['hospital_mortality'].sum():,}개 ({df_cohort['hospital_mortality'].mean()*100:.2f}%)")

## Step 2: DNR 시간 추출

In [ ]:
print("\nStep 2: DNR 시간 추출")

# stay_id 리스트를 DuckDB에 등록
con.register('cohort_df', df_cohort)

dnr_query = """
SELECT 
    c.stay_id,
    MIN(CAST(ce.charttime AS TIMESTAMP)) as dnr_time
FROM cohort_df c
INNER JOIN chartevents ce ON c.stay_id = ce.stay_id
WHERE ce.itemid = '223758'  -- Code Status / DNR
GROUP BY c.stay_id
"""

df_dnr = con.execute(dnr_query).df()
print(f"✓ DNR 기록 환자: {len(df_dnr):,}명")

# 코호트에 병합
df_cohort = df_cohort.merge(df_dnr, on='stay_id', how='left')

## Step 3: Ventilation 시작 시간 추출

In [ ]:
print("\nStep 3: Ventilation 시작 시간 추출")

vent_query = """
SELECT 
    c.stay_id,
    MIN(CAST(pe.starttime AS TIMESTAMP)) as vent_start_time
FROM cohort_df c
INNER JOIN procedureevents pe ON c.stay_id = pe.stay_id
WHERE pe.itemid = '225792'  -- Invasive Mechanical Ventilation
  AND pe.starttime IS NOT NULL
GROUP BY c.stay_id
"""

df_vent = con.execute(vent_query).df()
print(f"✓ Ventilation 기록 환자: {len(df_vent):,}명")

# 코호트에 병합
df_cohort = df_cohort.merge(df_vent, on='stay_id', how='left')

## Step 4: Pressor 시작 시간 추출

In [ ]:
print("\nStep 4: Pressor 시작 시간 추출")

pressor_query = """
SELECT 
    c.stay_id,
    MIN(CAST(ie.starttime AS TIMESTAMP)) as pressor_start_time
FROM cohort_df c
INNER JOIN inputevents ie ON c.stay_id = ie.stay_id
WHERE ie.itemid IN ('221906', '221289', '222315', '221662')  -- Norepinephrine, Epinephrine, Vasopressin, Dopamine
  AND ie.starttime IS NOT NULL
  AND ie.rate IS NOT NULL
  AND CAST(ie.rate AS DOUBLE) > 0
GROUP BY c.stay_id
"""

df_pressor = con.execute(pressor_query).df()
print(f"✓ Pressor 기록 환자: {len(df_pressor):,}명")

# 코호트에 병합
df_cohort = df_cohort.merge(df_pressor, on='stay_id', how='left')

## Step 5: 결과 확인 및 저장

In [ ]:
print("\n" + "="*60)
print("Cohort Base 요약")
print("="*60)

print(f"\n총 ICU stay 수: {len(df_cohort):,}개")
print(f"총 환자 수: {df_cohort['subject_id'].nunique():,}명")
print(f"평균 입실 횟수: {len(df_cohort) / df_cohort['subject_id'].nunique():.2f}회/환자")

print(f"\n=== 입실 유형 ===")
print(f"  - 첫 입실: {(df_cohort['is_readmission'] == 0).sum():,}개")
print(f"  - 재입실: {(df_cohort['is_readmission'] == 1).sum():,}개")

print(f"\n=== 이벤트 발생 stay ===")
print(f"  - DNR: {df_cohort['dnr_time'].notna().sum():,}개 ({df_cohort['dnr_time'].notna().mean()*100:.1f}%)")
print(f"  - Ventilation: {df_cohort['vent_start_time'].notna().sum():,}개 ({df_cohort['vent_start_time'].notna().mean()*100:.1f}%)")
print(f"  - Pressor: {df_cohort['pressor_start_time'].notna().sum():,}개 ({df_cohort['pressor_start_time'].notna().mean()*100:.1f}%)")
print(f"  - ICU 사망: {df_cohort['icu_mortality'].sum():,}개 ({df_cohort['icu_mortality'].mean()*100:.1f}%)")

print(f"\n=== 컬럼 목록 ({len(df_cohort.columns)}개) ===")
for col in df_cohort.columns:
    print(f"  - {col}")

In [ ]:
# CSV 저장
output_path = os.path.join(OUTPUT_DIR, 'cohort_base.csv')
df_cohort.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"\n✓ 저장 완료: cohort_base.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_cohort):,}개 (1 row = 1 stay)")
print(f"  - 환자 수: {df_cohort['subject_id'].nunique():,}명")
print(f"  - 경로: {output_path}")
print(f"\n⚠️ 주의: Train/Val/Test split 시 subject_id 기준으로 분리할 것!")

In [ ]:
# 샘플 데이터 확인
print("\n=== 샘플 데이터 (상위 5개) ===")
df_cohort.head()

In [ ]:
con.close()
print("\n=== 01. Cohort Base 생성 완료 ===")